<h2 style="font-family: 'Courier New';">
CAPSTONE - U.S STOCKS AND RETURNS
</h2>

**Capstone Project - U.S. Stock Market Analysisand Modeling**

Data source: [Stooq.com](https://stooq.com/t/?i=518&v=1) - a publicly accessible financial data platform covering U.S. listed stocks, scraped for   non-commercial purposes.

The dataset was scraped from Stooq.com (Index 518 - U.S. Stocks) and covers approximately 20,000+ publicly listed U.S. stocks across 3 separate data views.
It includes daily price data (Open, High, Low, Last closing price), trading Volume and Turnover, short to long-term return percentages (1D, 3D, 5D, 10D, 1M, 2M, 3M, 6M, YTD), and 52-week price metrics (52W Change, 1Y High, 1Y Low).

The goal of this project is to clean, segment, and analyze this data to understand stock behavior across liquidity, price, and return dimensions and then apply a machine learning algorithm to perform clustering,regression or classification.

Stooq does not prohibit non-commercial scraping in its `robots.txt` or Terms of Service, and all pages used are publicly accessible without login or authentication.

---

<h2 style="font-family: 'Courier New';">
Imports
</h2>

In [ ]:
import time, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# selenium and beautifulsoup imports

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC, color


---

<h2 style="font-family: 'Courier New';">
Web Scrapping script
</h2>

The scraper was built using **Selenium** (for JavaScript-rendered pages) and **BeautifulSoup** (for HTML parsing).
Key challenges included: handling pagination, multi-colspan table headers, hidden `<span>` elements corrupting percentage values, and delays.
Stooq splits data across 3 URL views (`v=1`, `v=4`, `v=6`), so each view was scraped separately and saved as its own CSV to keep scraping minimal and avoid rate limiting.

In [ ]:
'''

VIEWS refer to different pages in stooq website ( https://stooq.com/t/?i=518&v=1 , https://stooq.com/t/?i=518&v=4 ....)
handled by "v" parameter in the url:

- 1 = daily opening closing stock price data
- 4 = short to long term return from stocks
- 6 = 52 week change and year Hgh/Low stock price

INDEX_ID refer to the stock index were locking at example: US stocks which is handled by "i" parameter

PAGES refer to the No. of pages we would like to scrape each page contains 100 record which is handled by "l" :parameter in the url
( https://stooq.com/t/?i=518&v=4&l=2 ,https://stooq.com/t/?i=518&v=4&l=3 )

'''

BASE_URL = "https://stooq.com/t/"
INDEX_ID = 518
VIEWS = [1]
PAGES = 60


def parse_table(html):
    soup = BeautifulSoup(html, 'html.parser')
    table = soup.find('table', {'id': 'fth1'})
    if not table:
        return [], []
    for h in table.find_all(style="display:none"):
        h.decompose()
    rows = []

    for tr in table.find_all('tr'):
        row = []
        for cell in tr.find_all(['th', 'td']):
            row.append(cell.get_text(strip=True))
        if row:
            rows.append(row)

    return rows[0], rows[1:]


def run():
    opts = webdriver.ChromeOptions()
    opts.add_argument('--headless=new')
    driver = webdriver.Chrome(options=opts)

    driver.get('https://stooq.com/')
    time.sleep(3)  # let homepage JS challenge solve

    for view in VIEWS:
        all_rows = []
        headers = None

        for page in range(1, PAGES + 1):
            driver.get(f"{BASE_URL}?i={INDEX_ID}&v={view}&l={page}")

            try:
                WebDriverWait(driver, 15).until(
                    EC.presence_of_element_located((By.ID, 'fth1'))
                )
            except:
                print(f"v={view} page {page}: timed out, stopping")
                break

            time.sleep(1)

            h, rows = parse_table(driver.page_source)

            if headers is None and h:
                headers = h
                print(f"v={view} columns: {headers}")

            if not rows:
                print(f"v={view} page {page}: no data, stopping")
                break

            all_rows.extend(rows)
            print(f"v={view} page {page}: {len(rows)} rows (total {len(all_rows)})")

        if all_rows:
            n = len(headers)
            trimmed = [r[:n] for r in all_rows]
            df = pd.DataFrame(trimmed, columns=headers)
            df.to_csv(f'stooq_view{view}.csv', index=False, encoding='utf-8-sig')
            print(f"Saved stooq_view{view}.csv : {len(df)} rows\n")

    driver.quit()
    print("Done")


run()

After the scraper completes, it saves 3 CSV files:
- `stooq_view1.csv` — Daily price data: High, Low, Open, Last, Volume, Turnover, Change
- `stooq_view4.csv` — Short to long-term returns: 1D, 3D, 5D, 10D, 1M, 2M, 3M, 6M, YTD
- `stooq_view6.csv` — 52-week metrics: 52W Change, 1Y High, 1Y Low
